# Livrable 1 — Modélisation du problème de tournées de livraison

**Projet :** CesiCDP — réponse à l’appel à manifestation d’intérêt de l’ADEME  
**Problème retenu :** CVRP (Capacitated Vehicle Routing Problem), avec flotte homogène et dépôt unique.

> Ce notebook constitue le livrable de modélisation demandé au premier point de suivi. Il formalise le problème et sa complexité, puis propose une première formulation PLNE. Il ne traite pas du choix d’une méthode de résolution ni de son implémentation.

## 2. Données et modélisation mathématique en PLNE

Le réseau est un graphe complet orienté $G=(V,A)$, avec $C=\{1,\ldots,n\}$ les clients, $V=C\cup\{0\}$ le dépôt et les clients, $K=\{1,\ldots,m\}$ les véhicules, et $A=\{(i,j)\in V^2:i\ne j\}$ les arcs possibles. Chaque client $i$ a une demande $q_i>0$ avec $q_i\le Q$, chaque véhicule a une capacité $Q$, et $c_{ij}\ge0$ désigne le coût de déplacement de $i$ à $j$. Une condition nécessaire de faisabilité est $\sum_{i\in C}q_i\le mQ$.

Cette formulation utilise des variables **indexées par véhicule**. Cela permet de savoir non seulement quels arcs sont utilisés, mais aussi **quel camion** les emprunte. En contrepartie, il faut davantage de variables et de contraintes qu'avec une formulation qui ne distingue pas les véhicules. Elle s’inscrit dans le cadre général de la programmation linéaire présenté par [Techniques de l’ingénieur — Programmation linéaire](https://www.techniques-ingenieur.fr/ressources-documentaires/sciences-fondamentales-th8/mathematiques-ti052/programmation-lineaire-af1254) [2].

On note :
- $C=\{1,\ldots,n\}$ : ensemble des clients ;
- $V=C\cup\{0\}$ : clients et dépôt ;
- $K=\{1,\ldots,m\}$ : véhicules disponibles ;
- $A=\{(i,j)\in V^2:i\ne j\}$ : arcs possibles.

### 2.1 Variables de décision

- $x_{ijk}\in\{0,1\}$ : vaut 1 si le véhicule $k$ se déplace directement de $i$ vers $j$, et 0 sinon.
- $y_{ik}\in\{0,1\}$ : vaut 1 si le client $i$ est affecté au véhicule $k$.
- $z_k\in\{0,1\}$ : vaut 1 si le véhicule $k$ est utilisé.
- $u_{ik}\geq 0$ : quantité cumulée livrée par le véhicule $k$ jusqu'à sa visite du client $i$. Cette variable auxiliaire sert à modéliser la capacité et à éliminer les sous-tours.

**Pourquoi quatre familles de variables ?** Elles ne représentent pas la même décision : $x$ décrit les déplacements, $y$ l'affectation des clients, $z$ l'utilisation des camions et $u$ la progression de la charge cumulée. Une variable n'a pas besoin d'apparaître dans la fonction objectif pour être utile : elle peut servir uniquement aux contraintes.

### 2.2 Fonction objectif

On minimise le coût total des arcs empruntés par tous les véhicules :

$$\min Z=\sum_{k\in K}\sum_{(i,j)\in A}c_{ij}x_{ijk}$$

Ici, $c_{ij}$ représente la distance (en km) ou la durée (en minutes) entre $i$ et $j$. Comme $x_{ijk}=1$ uniquement si le véhicule $k$ emprunte cet arc, la somme additionne les coûts de toutes les tournées.

### 2.3 Contraintes

#### A. Chaque client est affecté à un seul véhicule

$$\sum_{k\in K}y_{ik}=1 \qquad \forall i\in C$$

Pour chaque client $i$, exactement un véhicule est choisi. Cette contrainte porte sur **l'affectation** des clients, pas directement sur les arcs.

#### B. Un client affecté au véhicule doit avoir une entrée et une sortie

$$\sum_{j\in V\setminus\{i\}}x_{ijk}=y_{ik}
\qquad \forall i\in C,\ k\in K$$

$$\sum_{j\in V\setminus\{i\}}x_{jik}=y_{ik}
\qquad \forall i\in C,\ k\in K$$

Si $y_{ik}=1$, le véhicule $k$ a exactement un arc sortant du client $i$ et un arc entrant dans ce client. Si $y_{ik}=0$, ces sommes valent 0 : ce véhicule ne visite pas ce client.

#### C. Chaque véhicule utilisé part du dépôt et y revient

$$\sum_{j\in C}x_{0jk}=z_k \qquad \forall k\in K$$

$$\sum_{i\in C}x_{i0k}=z_k \qquad \forall k\in K$$

Si $z_k=1$, le véhicule $k$ a un départ du dépôt et un retour au dépôt. Si $z_k=0$, il n'effectue aucun de ces déplacements.

#### D. Relier l'affectation à l'utilisation du véhicule

$$y_{ik}\leq z_k \qquad \forall i\in C,\ k\in K$$

Un véhicule inutilisé ($z_k=0$) ne peut pas se voir affecter de client. Cette contrainte rend explicite le lien entre les variables $y$ et $z$.

#### E. Respecter la capacité du véhicule

$$\sum_{i\in C}q_i y_{ik}\leq Qz_k \qquad \forall k\in K$$

La demande totale des clients affectés à un véhicule ne dépasse pas sa capacité $Q$. Si le véhicule n'est pas utilisé, $z_k=0$ et la contrainte impose qu'aucun client ne lui soit affecté.

#### F. Cohérence de charge et élimination des sous-tours

On borne d'abord la charge cumulée :

$$q_i y_{ik}\leq u_{ik}\leq Qy_{ik}
\qquad \forall i\in C,\ k\in K$$

Si le client $i$ n'est pas affecté au véhicule $k$, alors $y_{ik}=0$ et donc $u_{ik}=0$. S'il lui est affecté, la charge cumulée est au moins égale à la demande de ce client et ne dépasse pas $Q$. Le terme $q_j y_{jk}$ est essentiel : il désactive l’ajout de la demande de $j$ lorsque $j$ n’appartient pas au véhicule $k$.

Ensuite, pour deux clients distincts $i$ et $j$ :

$$u_{jk}\geq u_{ik}+q_j y_{jk}-Q(1-x_{ijk})
\qquad \forall k\in K,\ i,j\in C,\ i\ne j$$

**Si l'arc $i\to j$ est emprunté**, $x_{ijk}=1$ et la contrainte devient :

$$u_{jk}\geq u_{ik}+q_j y_{jk}=u_{ik}+q_j$$

La quantité cumulée augmente donc au moins de la demande $q_j$ du client suivant.

**Si l'arc $i\to j$ n'est pas emprunté**, $x_{ijk}=0$ et la contrainte devient $u_{jk}\geq u_{ik}+q_j y_{jk}-Q$, une borne valide car $0\leq u_{ik}\leq Q$ et $q_j y_{jk}\leq Q$.

La contrainte est relâchée lorsque l’arc n’est pas utilisé ; le terme d’activation doit inclure $y_{jk}$ pour rester valide entre véhicules.

Cette progression empêche un sous-tour formé uniquement de clients : si un cycle de clients était utilisé, la charge cumulée devrait augmenter d'une quantité strictement positive à chaque arc, puis revenir à son point de départ, ce qui est impossible.

#### G. Domaine des variables

$$x_{ijk},y_{ik},z_k\in\{0,1\},\qquad u_{ik}\geq 0$$

Les variables $x$, $y$ et $z$ sont binaires, tandis que $u$ est continue. Elle n'a pas besoin d'être entière pour que cette formulation fonctionne.

### 2.4 Limites et prolongements

Cette formulation suppose des coûts d’arcs fixes. Elle ne comprend pas de fenêtres horaires, de durée maximale de tournée, ni de trafic dépendant de l’heure. Ces extensions demanderaient des variables et contraintes supplémentaires.


## 3. Analyse théorique de complexité

Maintenant que le problème et sa formulation mathématique en PLNE sont établis, nous pouvons en analyser la complexité algorithmique intrinsèque.

### 3.1 Version décisionnelle du CVRP

La version d'optimisation du CVRP cherche le coût minimum $Z^*$. La **version décisionnelle** associée s'énonce ainsi :
> **Entrée :** Un graphe complet $G=(V, A)$, un ensemble de clients $C$ avec demandes $q_i$, un dépôt $0$, une capacité $Q$, une flotte maximale $m$, une matrice de coûts $c_{ij}$, et un seuil de coût rationnel $B > 0$.  
> **Question :** Existe-t-il un ensemble de tournées satisfaisant les contraintes du modèle (chaque client affecté à un seul véhicule, départ et retour au dépôt, au plus $m$ véhicules utilisés et respect de la capacité $Q$) et de coût total vérifiant $\sum_{k\in K}\sum_{(i,j)\in A} c_{ij}x_{ijk} \le B$ ?

---

### 3.2 Appartenance à la classe NP

Pour établir que le CVRP décisionnel appartient à la classe **NP**, on fournit comme certificat une liste de tournées ordonnées $\mathcal{R}=(R_1,\ldots,R_p)$, où chaque tournée est de la forme $R_k=(0,i_1,\ldots,i_r,0)$. Les véhicules sont identifiés par ces tournées ; il suffit ensuite de vérifier :

1. **Nombre de véhicules :** vérifier que $p\le m$.
2. **Départ et retour au dépôt :** vérifier que chaque tournée commence au dépôt $0$ et s'y termine.
3. **Visite unique :** vérifier que chaque client de $C$ apparaît exactement une fois dans l'ensemble des tournées.
4. **Capacité :** pour chaque tournée $R_k$, calculer la somme des demandes de ses clients et vérifier qu'elle est au plus $Q$.
5. **Seuil de coût :** additionner les coûts des arcs parcourus et vérifier que le coût total est au plus $B$.

Chaque contrôle se fait en temps polynomial dans la taille de l'entrée (linéaire en nombre de clients et de tournées, hors coût des opérations arithmétiques). Le certificat a lui-même une taille polynomiale. Les variables $x_{ijk}$, $y_{ik}$ et $z_k$ de la PLNE peuvent être déduites de ces tournées. Ainsi :

$$\boxed{\text{Le CVRP décisionnel appartient à la classe NP.}}$$

---

### 3.3 Réduction du TSP au CVRP
#### 3.3.1 Partir d’une instance du TSP

On considère une instance du TSP définie par :
- Un ensemble de villes $V=\{v_0,v_1,\ldots,v_n\}$.
- Un coût de déplacement $c_{ij}$ entre chaque paire de villes.
- Une borne de coût $B$.

#### 3.3.2 Construire une instance du CVRP
  
On transforme l'instance du TSP de la manière suivante :

| Instance du TSP | Instance correspondante du CVRP |
|---|---|
| Ville $v_0$ | Dépôt $v_0$ |
| Les $n$ villes restantes | Les $n$ sommets restants deviennent les clients $C$ |
| Coût de déplacement $c_{ij}$ | Même coût de déplacement $c_{ij}$ |
| Aucune demande des clients | Fixer la demande de chaque client à $q_i=1$ |
| Un seul véhicule | Fixer le nombre de véhicules à $m=1$ |
| Aucune contrainte de capacité | Fixer la capacité du véhicule à $Q=n$ |
| Borne de coût $B$ | Même borne de coût $B$ |

**Remarquons que cette transformation s'effectue en temps polynomial.**

#### 3.3.3 Démontrer l’équivalence des réponses

4.3.1. Si l'instance du TSP admet une tournée de coût inférieur ou égal à $B$, alors l'instance du CVRP admet également une solution de coût inférieur ou égal à $B$.

On fait suivre au véhicule du CVRP la tournée correspondant à la solution du TSP.
Cette tournée :
- Dessert chaque client exactement une fois.
- Respecte la capacité du véhicule $Q=n$.
- Commence et se termine au dépôt.
- Possède exactement le même coût total que la tournée du TSP.

Par conséquent, il s'agit d'une solution réalisable du CVRP dont le coût est inférieur ou égal à $B$.

4.3.2. Si l'instance du CVRP construite admet une solution de coût inférieur ou égal à $B$, alors l'instance du TSP admet également une tournée de coût inférieur ou égal à $B$.

Il n'y a qu'un seul véhicule effectuant une seule tournée, et chaque client doit être desservi exactement une fois.
Cette tournée satisfait donc les contraintes du TSP et possède un coût inférieur ou égal à $B$.

Puisque le TSP décisionnel est NP-complet, cette réduction montre que le CVRP décisionnel est NP-difficile. Avec son appartenance à NP établie à la section précédente, le CVRP décisionnel est donc **NP-complet**. Le problème d’optimisation CVRP associé est **NP-difficile**.

---

### 3.4 Conséquences théoriques et pratiques

1. **Absence d'algorithme polynomial exact :** Sauf si $P = NP$, il n'existe aucun algorithme capable de résoudre de façon exacte toutes les instances du CVRP en temps polynomial.
2. **Explosion combinatoire :** Le nombre de permutations possibles pour ordonner $n$ clients est $n!$. Même avec l'élagage efficace du Branch and Bound sur le modèle PLNE, la complexité dans le pire cas reste exponentielle.


## 4. Références

### Ressources fournies

[P versus NP : exemple dans un réseau social](https://www.youtube.com/watch?v=rkDlTQFucBM), Rachid Guerraoui, vidéo YouTube. — **Complexité et distinction P/NP**.

[Programmation linéaire](https://www.techniques-ingenieur.fr/ressources-documentaires/sciences-fondamentales-th8/mathematiques-ti052/programmation-lineaire-af1254), Techniques de l’ingénieur. — **Cadre de la programmation linéaire**.

[Programmation linéaire et PuLP](https://scenari.cesi.fr/scenari/cas/Diffusion_FISE_INFO_A3_Algorithmique_et_optimisation_26_27_ETU/res/PRogLineMaire_Et_Pulp_V1.pdf), support pédagogique CESI. — **Passage du modèle à l’implémentation**.

[La modélisation — Recherche opérationnelle](https://www.youtube.com/watch?v=Zy-iTwA9tfU), vidéo YouTube. — **Démarche de modélisation**.